# Coherent info

In [1]:
import numpy as np
from coherentinfo.moebius_two_odd_prime import MoebiusCodeTwoOddPrime
from coherentinfo.postprocess import (aggregate_data,
                                      aggregate_data_jax,
                                      compute_conditional_entropy_term,
                                      miller_madow_conditional_entropy)


from coherentinfo.errormodel import ErrorModelLindbladTwoOddPrime
# import galois
import scipy
import timeit
import jax
import jax.numpy as jnp

In [2]:
length = 3
width = 3
p = 3
moebius_code = MoebiusCodeTwoOddPrime(length=length, width=width, d=2 * p)
# moebius_code_qubit = MoebiusCodeQubit(length=length, width=width)
h_z = moebius_code.h_z
h_x = moebius_code.h_x
logical_x = moebius_code.logical_x
logical_z = moebius_code.logical_z

In [3]:
gamma = 0.3
em_lindblad = ErrorModelLindbladTwoOddPrime(
    moebius_code.num_edges, d=2 * p, gamma_t=gamma
)

In [14]:
num_samples = 10_000_000
# vertex_result = moebius_code_qubit.compute_batched_vertex_syndrome_chi_z(
#     num_samples, em_moebius_qubit_jax)
base_key = jax.random.PRNGKey(0)
vertex_result = moebius_code.compute_batched_vertex_syndrome_chi_z(
    num_samples, em_lindblad, base_key)

In [15]:
aggregate_data_jit = jax.jit(
    aggregate_data_jax, static_argnums=(1,))
vertex_pads = -1 * jnp.ones(vertex_result.shape[1] - 1)

_, vertex_counts = aggregate_data_jit(
    vertex_result,
    num_samples,  # Passed as the static (Python int) argument
    vertex_pads
)
del vertex_result
del vertex_pads

In [16]:
def counts_to_prob(count):
    total_count = jnp.sum(count)
    result = jax.lax.cond(total_count == 0, lambda: count.astype(
        float), lambda: count / total_count)
    return result


vertex_probs = jax.vmap(counts_to_prob)(vertex_counts)
vertex_probs

Array([[0.7277061 , 0.27229396],
       [0.62653095, 0.37346902],
       [0.5478022 , 0.4521978 ],
       ...,
       [0.        , 0.        ],
       [0.        , 0.        ],
       [0.        , 0.        ]], dtype=float32)

In [17]:
vertex_conditional_entropy = miller_madow_conditional_entropy(
    vertex_probs, num_samples)
# del vertex_probs
print(vertex_conditional_entropy)

0.4946796


In [18]:
base_key = jax.random.PRNGKey(45)
plaquette_result = moebius_code.compute_batched_plaquette_syndrome_chi_x(
    num_samples, em_lindblad, base_key)

In [19]:
plaquette_pads = -1 * jnp.ones(plaquette_result.shape[1] - 1)
_, plaquette_counts = aggregate_data_jit(
    plaquette_result,
    num_samples,  # Passed as the static (Python int) argument
    plaquette_pads
)
del plaquette_result
del plaquette_pads

In [20]:
plaquette_probs = jax.vmap(counts_to_prob)(plaquette_counts)

In [21]:
plaquette_conditional_entropy = miller_madow_conditional_entropy(
    plaquette_probs, num_samples)
del plaquette_probs

In [22]:
print(plaquette_conditional_entropy)

0.09981678


In [23]:
coherent_info = 1.0 - vertex_conditional_entropy - plaquette_conditional_entropy
print("Coherent information = {}".format(coherent_info))

Coherent information = 0.40550366044044495
